# PATH Step 2: Component Network Model with Prognostic Risk

This notebook implements the second stage of the PATH-style analysis.

The aim is to estimate:

1. Average effects of collaborative-care treatment components.
2. The association between prognostic risk and PHQ-9 outcome.
3. Whether the effect of each treatment compoMnent varies according to prognostic risk.

The intended model is conceptually:

$$
\Delta PHQ9_{ij}
=
\alpha_j
+
\lambda R_{ij}
+
\sum_{k=1}^{K}\beta_k C_{ijk}
+
\sum_{k=1}^{K}\delta_k C_{ijk}R_{ij}
+
\epsilon_{ij}
$$

where:

- $\Delta PHQ9_{ij}$ is 12-month PHQ-9 minus baseline PHQ-9.
- $\alpha_j$ is the study-specific intercept.
- $R_{ij}$ is the patient's prognostic score.
- $C_{ijk}$ indicates whether component $k$ is present.
- $\beta_k$ is the average component effect.
- $\delta_k$ describes how the component effect changes across prognostic risk.

Negative PHQ-9 change indicates improvement.

## Temporary prognostic-score placeholder

The final prognostic score from PATH Step 1 is not yet available.

For development purposes only, baseline PHQ-9 is temporarily used as the prognostic-score placeholder:

$$
R_i = PHQ9_{baseline,i}
$$

This should be replaced by the final multivariable prognostic score once Step 1 is complete.

The prognostic score is currently kept on its original scale. Centering or standardizing it may later be considered to improve interpretation of the component main effects.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

IPD_DATA_PATH = PROCESSED_DIR / "enriched_dataset.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", IPD_DATA_PATH)
print("Dataset exists:", IPD_DATA_PATH.exists())

## 1. Load the harmonized IPD dataset

The enriched dataset contains the harmonized individual-participant data from the available collaborative-care trials.

The dataset is longitudinal, so each patient may have multiple rows corresponding to different follow-up time points.

In [ ]:
df = pd.read_csv(
    IPD_DATA_PATH,
    low_memory=False,
)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Studies:", df["STUDY_ID"].nunique())
print("Patients:", df["patient_id"].nunique())

## 2. Define the treatment components

These are the 13 collaborative-care components retained after the component-validation stage.

Component variables describe the planned treatment package and are therefore treated as treatment-assignment characteristics rather than patient characteristics.

In [ ]:
component_cols = [
    "treatment_counseling_was_provided_in_addition_to_medication_(≤ 8 sessions)",
    "treatment_manual_based_psychotherapy",
    "treatment_treatment_goals_were_considered",
    "treatment_patient_preference_was_considered_in_treatment_decisions",
    "treatment_follow_up_frequency_every_(2-4 weeks)",
    "treatment_follow_up_frequency_( ≤2 weeks)",
    "treatment_form_of_monitoring",
    "treatment_automated_follow_up_process_was_used",
    "treatment_adhoc_emergency_comm",
    "treatment_is_there_a_shared_medical_record",
    "treatment_are_disease_coping_strategies_taught",
    "treatment_is_there_a_relapse_prevention_plan",
    "treatment_were_family_and_friends_involved",
]

print("Number of components:", len(component_cols))

In [ ]:
missing_component_cols = [
    col for col in component_cols
    if col not in df.columns
]

print("Missing component columns:")
print(missing_component_cols)

## 3. Identify the age and sex variables

The Step 2 analysis will initially be restricted to patients with recorded age and sex/gender.

Although age and sex will not be entered separately into the PATH risk-score model once the final prognostic score is available, they are required here because they are expected to contribute to the Step 1 prognostic model.

For the temporary analysis, baseline PHQ-9 will be used as the prognostic-score placeholder.

In [ ]:
if "sex" in df.columns:
    sex_col = "sex"
elif "gender" in df.columns:
    sex_col = "gender"
else:
    raise KeyError(
        "Could not find either 'sex' or 'gender' in the dataset."
    )

print("Sex/gender column:", sex_col)

In [ ]:
if "age" not in df.columns:
    raise KeyError("Column 'age' was not found.")

print("Age column found.")

## 4. Prepare key variables

PHQ-9, follow-up month, and age are converted to numeric values.

Invalid or non-numeric values are converted to missing values so that they can be excluded explicitly during construction of the analysis sample.

In [ ]:
df["follow_up_months"] = pd.to_numeric(
    df["follow_up_months"],
    errors="coerce",
)

df["phq9_total"] = pd.to_numeric(
    df["phq9_total"],
    errors="coerce",
)

df["age"] = pd.to_numeric(
    df["age"],
    errors="coerce",
)

## 5. Restrict to patients with age and sex/gender information

Patients without recorded age or sex/gender are excluded from this preliminary Step 2 analysis.

Because the dataset is longitudinal, the filtering below initially operates on rows. Patient-level eligibility will be finalized after baseline and 12-month observations are paired.

In [ ]:
eligible_demo = df[
    df["age"].notna()
    & df[sex_col].notna()
].copy()

print("Rows before demographic restriction:", len(df))
print("Rows after requiring age + sex:", len(eligible_demo))

print(
    "Patients remaining:",
    eligible_demo[
        ["STUDY_ID", "patient_id"]
    ].drop_duplicates().shape[0],
)

print(
    "Studies remaining:",
    eligible_demo["STUDY_ID"].nunique(),
)

## 6. Construct the baseline PHQ-9 dataset

Baseline is defined as:

$$
t = 0
$$

Patients must have a non-missing PHQ-9 score at baseline.

For now, baseline PHQ-9 also serves as the temporary prognostic score.

In [ ]:
missing_phq9 = eligible_demo["phq9_total"].isna().sum()
print("Rows without PHQ-9 (removed):", missing_phq9)

has_phq9 = eligible_demo[eligible_demo["phq9_total"].notna()].copy()

print("Studies remaining:", has_phq9["STUDY_ID"].nunique())
print("Unique patients remaining:", has_phq9["patient_id"].nunique())

In [ ]:
patients_with_baseline = has_phq9.loc[
    has_phq9["follow_up_months"] == 0, "patient_id"
].unique()

no_baseline = has_phq9[
    ~has_phq9["patient_id"].isin(patients_with_baseline)
]

print("Patients with no follow_up_months == 0 row:", no_baseline["patient_id"].nunique())
print("Studies affected:", no_baseline["STUDY_ID"].nunique())

In [ ]:
for study, patient in no_baseline[["STUDY_ID", "patient_id"]].drop_duplicates().itertuples(index=False):
    print(study, patient)

In [ ]:
baseline_duplicates = baseline.duplicated(
    subset=["STUDY_ID", "patient_id"],
    keep=False,
)

print(
    "Patients with duplicate baseline rows:",
    baseline_duplicates.sum(),
)

## 7. Construct the 12-month PHQ-9 dataset

The outcome requires a PHQ-9 measurement at 12 months.

Patients without a valid 12-month PHQ-9 observation will not enter the final Step 2 analysis.

In [ ]:
followup_12 = (
    eligible_demo[
        (eligible_demo["follow_up_months"] == 12)
        & eligible_demo["phq9_total"].notna()
        & eligible_demo["study_arm"].isin(
            ["control", "intervention"]
        )
    ][
        [
            "STUDY_ID",
            "patient_id",
            "phq9_total",
        ]
    ]
    .rename(
        columns={
            "phq9_total": "phq9_12m"
        }
    )
    .copy()
)

print("12-month observations:", len(followup_12))
print("12-month studies:", followup_12["STUDY_ID"].nunique())

In [ ]:
base = eligible_demo[
    eligible_demo["phq9_total"].notna()
    & eligible_demo["study_arm"].isin(["control", "intervention"])
]

window = base[
    (base["follow_up_months"] >= 6) & (base["follow_up_months"] <= 12)
]

for study_id, group in window.groupby("STUDY_ID"):
    total_patients = base.loc[
        base["STUDY_ID"] == study_id, "patient_id"
    ].nunique()

    print(f"\nStudy: {study_id} (total patients w/ PHQ-9, control/intervention: {total_patients})")

    value_counts = group.groupby("follow_up_months")["patient_id"].nunique().sort_index()
    for value, count in value_counts.items():
        pct = 100 * count / total_patients
        print(f"  follow_up_months = {value}: {count} patients ({pct:.1f}%)")

In [ ]:
base = eligible_demo[
    eligible_demo["phq9_total"].notna()
    & eligible_demo["study_arm"].isin(["control", "intervention"])
]

window = base[
    (base["follow_up_months"] >= 6) & (base["follow_up_months"] <= 12)
]

for study_id in sorted(base["STUDY_ID"].unique()):
    total_patients = base.loc[
        base["STUDY_ID"] == study_id, "patient_id"
    ].nunique()

    print(f"\nStudy: {study_id} (total patients w/ PHQ-9, control/intervention: {total_patients})")

    study_window = window[window["STUDY_ID"] == study_id]

    if study_window.empty:
        print("  No patients with follow_up_months between 6 and 12.")
        continue

    value_counts = study_window.groupby("follow_up_months")["patient_id"].nunique().sort_index()
    for value, count in value_counts.items():
        pct = 100 * count / total_patients
        print(f"  follow_up_months = {value}: {count} patients ({pct:.1f}%)")

In [ ]:
no_window_studies = ["11_Fletcher_2021b", "21_Richards_2008", "25_Rollman_2017"]

for study_id in no_window_studies:
    study_data = base[base["STUDY_ID"] == study_id]

    print(f"\nStudy: {study_id}")
    value_counts = (
        study_data.groupby("follow_up_months")["patient_id"]
        .nunique()
        .sort_index()
    )
    total_patients = study_data["patient_id"].nunique()
    for value, count in value_counts.items():
        pct = 100 * count / total_patients
        print(f"  follow_up_months = {value}: {count} patients ({pct:.1f}%)")